<a href="https://colab.research.google.com/github/mshitie/Dual-Brach/blob/main/quantitative%20XAI%20evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DenseNet121 + Quantitative XAI on BUSI Breast Ultrasound Dataset
Classes: benign, malignant, normal
XAI methods: Grad-CAM, Integrated Gradients (+ Random-map baseline for sanity)
Quantitative metrics:
  Faithfulness : Deletion AUC (lower = better), Insertion AUC (higher = better),
                 Average Drop % (lower = better), Increase in Confidence % (higher = better)
  Localization : Lesion-mask energy, rank accuracy, pointing game, IoU (uses dataset masks;
                 computed on benign/malignant images only - normal images have empty masks)
  Stability    : Spearman correlation of maps under small Gaussian noise

In [ ]:
! pip install -q kaggle

In [ ]:
! cp kaggle.json ~/.kaggle/
#!cp /content/drive/MyDrive/computer_vision/kaggle.json ~/.kaggle/kaggle.json # if you upload from gdrive

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR']= '/content'

In [ ]:
!kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset
!unzip -q breast-ultrasound-images-dataset.zip -d busi

In [ ]:
import os, glob, random, warnings
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input   # ImageNet mean/std normalisation (torch-style)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, auc
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = "/content/busi/Dataset_BUSI_with_GT"   # <-- change to your path (contains benign/ malignant/ normal/)
CLASSES = ["benign", "malignant", "normal"]
IMG_SIZE, BATCH = 224, 32
MAX_PER_CLASS = None      # e.g. 1500 to balance classes / speed up training
N_EVAL_PER_CLASS = 25     # images per class used for the XAI evaluation
# Neutral-gray image in the model's input space (used as XAI baseline / 'deleted pixel' value)
# and the input-space change per +1 raw pixel level (used to scale noise in the stability test).
GRAY = preprocess_input(np.full((IMG_SIZE, IMG_SIZE, 3), 127.5, "float32"))
SCALE = preprocess_input(np.full((IMG_SIZE, IMG_SIZE, 3), 128.5, "float32")) - GRAY
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
rows = []
for c in CLASSES:
    for p in sorted(glob.glob(os.path.join(DATA_DIR, c, "*.png"))):
        if "_mask" in os.path.basename(p):          # masks live in the same folder as the images
            continue
        # a few images have several masks: "x (1)_mask.png", "x (1)_mask_1.png", ...
        masks = sorted(glob.glob(glob.escape(p[:-4]) + "_mask*.png"))
        rows.append({"path": p, "mask_path": "|".join(masks), "label": c})
df = pd.DataFrame(rows)

if MAX_PER_CLASS:
    df = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED)
                    for _, g in df.groupby("label")]).reset_index(drop=True)

label2idx = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(label2idx)
print(df["label"].value_counts())

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df.y, random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df.y, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
aug = tf.keras.Sequential([layers.RandomRotation(0.04),
                           layers.RandomZoom(0.1),
                           layers.RandomTranslation(0.05, 0.05)])   # no horizontal flip (heart side matters)

def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    return tf.image.resize(img, (IMG_SIZE, IMG_SIZE))               # float32, 0-255

def make_ds(d, training=False):
    ds = tf.data.Dataset.from_tensor_slices((d.path.values, d.y.values))
    if training:
        ds = ds.shuffle(len(d), seed=SEED)
    ds = ds.map(lambda p, y: (load_img(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH)
    if training:
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)


In [ ]:
base = DenseNet121(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(len(CLASSES), activation="softmax")(x)
model = models.Model(base.input, out, name="densenet121_cxr")

def set_trainable(unfreeze_top):
    # Backbone frozen in phase 1; in phase 2 only the last dense block (conv5_*) is unfrozen.
    # BatchNorm layers always stay frozen (stable fine-tuning with small LR / batch).
    for l in base.layers:
        l.trainable = (unfreeze_top and l.name.startswith("conv5")
                       and not isinstance(l, layers.BatchNormalization))

def compile_model(lr):
    model.compile(optimizer=optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cw = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=train_df.y.values)
cw = {i: w for i, w in enumerate(cw)}
cbs = [callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_loss"),
       callbacks.ReduceLROnPlateau(factor=0.3, patience=2, monitor="val_loss")]

# Phase 1: train the head only
set_trainable(False); compile_model(1e-3)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=cw, callbacks=cbs)
# Phase 2: fine-tune the last dense block (conv5)
set_trainable(True); compile_model(1e-5)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=cw, callbacks=cbs)
model.save("densenet121_covid_cxr.keras")


In [ ]:
y_prob = model.predict(test_ds)
y_pred = y_prob.argmax(1)
y_true = test_df.y.values
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print("Macro OvR AUC:", roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()


In [ ]:
# last conv feature map of DenseNet121 (7x7x1024, after final BN+ReLU) = backbone output
grad_model = models.Model(model.input, [base.output, model.output])

def minmax(a):
    a = a - a.min()
    return a / (a.max() + 1e-8)

def gradcam(x, cls=None):
    """x: (1,224,224,3) preprocessed. Returns map in [0,1] (224x224)."""
    if cls is None:
        cls = int(np.argmax(model(x, training=False)[0]))
    with tf.GradientTape() as tape:
        conv, preds = grad_model(x, training=False)
        score = preds[:, cls]
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0].numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    return minmax(cam)

def integrated_gradients(x, cls=None, steps=32):
    if cls is None:
        cls = int(np.argmax(model(x, training=False)[0]))
    x = tf.convert_to_tensor(x, tf.float32)
    baseline = tf.constant(GRAY[None], tf.float32)   # neutral gray image (model-input space)
    alphas = tf.linspace(0.0, 1.0, steps + 1)[:, None, None, None]
    inter = baseline + alphas * (x - baseline)
    with tf.GradientTape() as tape:
        tape.watch(inter)
        p = model(inter, training=False)[:, cls]
    g = tape.gradient(p, inter)
    avg = tf.reduce_mean((g[:-1] + g[1:]) / 2.0, axis=0, keepdims=True)   # trapezoid rule
    ig = ((x - baseline) * avg)[0]
    attr = tf.reduce_sum(tf.abs(ig), axis=-1).numpy()
    attr = cv2.GaussianBlur(attr, (0, 0), 2)
    return minmax(attr)

def random_map(x, cls=None):
    return minmax(cv2.GaussianBlur(np.random.rand(IMG_SIZE, IMG_SIZE).astype("float32"), (0, 0), 15))

EXPLAINERS = {"Grad-CAM": gradcam, "Integrated Gradients": integrated_gradients,
              "Random (baseline)": random_map}


In [ ]:
def predict_cls_prob(imgs, cls):
    return model.predict(imgs, batch_size=32, verbose=0)[:, cls]

def deletion_insertion(x, cam, cls, steps=20):
    """x: (224,224,3) preprocessed. Returns (deletion_auc, insertion_auc, curves)."""
    order = np.argsort(-cam.ravel())
    n = cam.size
    blur = cv2.GaussianBlur(x, (0, 0), 10)
    fr = np.linspace(0, 1, steps + 1)
    del_imgs, ins_imgs = [], []
    for f in fr:
        mask = np.zeros(n, bool); mask[order[:int(f * n)]] = True
        mask = mask.reshape(IMG_SIZE, IMG_SIZE, 1)
        del_imgs.append(np.where(mask, GRAY, x))     # remove most relevant pixels -> neutral gray
        ins_imgs.append(np.where(mask, x, blur))     # reveal most relevant pixels on blurred image
    pd_ = predict_cls_prob(np.stack(del_imgs), cls)
    pi_ = predict_cls_prob(np.stack(ins_imgs), cls)
    return auc(fr, pd_), auc(fr, pi_), (fr, pd_, pi_)

def drop_and_increase(x, cam, cls):
    """Average Drop & Increase-in-Confidence: explanation-masked image vs. original."""
    y = predict_cls_prob(x[None], cls)[0]
    o = predict_cls_prob((GRAY + (x - GRAY) * cam[..., None])[None], cls)[0]   # fade low-relevance pixels to gray
    return max(0.0, y - o) / (y + 1e-8) * 100, float(o > y) * 100

def lesion_mask(paths):
    """Union of all mask files of an image ("|"-separated); all zeros for normal images."""
    m = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
    for p in str(paths).split("|"):
        if p:
            mm = cv2.resize(cv2.imread(p, 0), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            m = np.maximum(m, (mm > 127).astype(np.float32))
    return m

def localization(cam, mask):
    energy = (cam * mask).sum() / (cam.sum() + 1e-8)                      # relevance mass inside lesion
    K = int(mask.sum())
    top = np.argsort(-cam.ravel())[:K]
    rank_acc = mask.ravel()[top].mean()                                   # top-K pixels inside lesion
    pointing = float(mask.ravel()[np.argmax(cam.ravel())] > 0)            # max point inside lesion
    binm = cam > 0.5
    iou = (binm & (mask > 0)).sum() / ((binm | (mask > 0)).sum() + 1e-8)
    return energy, rank_acc, pointing, iou

def stability(explainer, x, cls, base_cam, n=3, sigma=5.0):
    """Mean Spearman correlation between the map and maps from noisy inputs
    (Gaussian noise with sigma=5 on the raw 0-255 pixel scale)."""
    rs = []
    for _ in range(n):
        xn = x + (np.random.normal(0, sigma, x.shape) * SCALE).astype("float32")
        if int(np.argmax(model(xn[None], training=False)[0])) != cls:
            continue
        c2 = explainer(xn[None], cls)
        rs.append(spearmanr(base_cam.ravel(), c2.ravel())[0])
    return np.nanmean(rs) if rs else np.nan


In [ ]:
eval_df = pd.concat([g.sample(min(len(g), N_EVAL_PER_CLASS), random_state=SEED)
                     for _, g in test_df.groupby("label")]).reset_index(drop=True)

records, curves = [], {k: {"del": [], "ins": []} for k in EXPLAINERS}
for i, r in eval_df.iterrows():
    x = preprocess_input(load_img(r["path"]).numpy())
    cls = int(np.argmax(model(x[None], training=False)[0]))
    mask = lesion_mask(r["mask_path"])
    for name, fn in EXPLAINERS.items():
        cam = fn(x[None], cls)
        d_auc, i_auc, (fr, pdel, pins) = deletion_insertion(x, cam, cls)
        avg_drop, inc_conf = drop_and_increase(x, cam, cls)
        if mask.sum() > 0:
            energy, rank_acc, pointing, iou = localization(cam, mask)
        else:                                  # normal image: no lesion -> localization undefined
            energy = rank_acc = pointing = iou = np.nan
        stab = stability(fn, x, cls, cam)
        curves[name]["del"].append(pdel); curves[name]["ins"].append(pins)
        records.append(dict(method=name, true=r.label, pred=CLASSES[cls], correct=cls == r.y,
                            deletion_auc=d_auc, insertion_auc=i_auc,
                            avg_drop=avg_drop, inc_conf=inc_conf,
                            lesion_energy=energy, rank_acc=rank_acc, pointing=pointing, iou=iou,
                            stability=stab, lesion_area_frac=mask.mean()))
    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(eval_df)} images done")

res = pd.DataFrame(records)
res.to_csv("xai_quantitative_per_image.csv", index=False)


In [ ]:
metric_cols = ["deletion_auc", "insertion_auc", "avg_drop", "inc_conf",
               "lesion_energy", "rank_acc", "pointing", "iou", "stability"]
summary = res.groupby("method")[metric_cols].agg(["mean", "std"]).round(4)
print("=== Overall (mean, std) ===")
print(summary.T)
summary.to_csv("xai_quantitative_summary.csv")

print("\n=== Per true class (mean) ===")
per_class = res.groupby(["method", "true"])[metric_cols].mean().round(4)
print(per_class)
per_class.to_csv("xai_quantitative_per_class.csv")

print("\nChance level for lesion_energy / rank_acc / pointing ~ mean lesion area fraction:",
      round(res.loc[res.lesion_area_frac > 0, "lesion_area_frac"].mean(), 3))
print("Interpretation: deletion_auc & avg_drop lower = better | insertion_auc, inc_conf, "
      "lesion_energy, rank_acc, pointing, iou, stability higher = better.")


In [ ]:
fr = np.linspace(0, 1, 21)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name in EXPLAINERS:
    ax[0].plot(fr, np.mean(curves[name]["del"], 0), label=name)
    ax[1].plot(fr, np.mean(curves[name]["ins"], 0), label=name)
ax[0].set_title("Deletion (steeper drop = better)"); ax[1].set_title("Insertion (steeper rise = better)")
for a in ax:
    a.set_xlabel("Fraction of pixels"); a.set_ylabel("Class probability"); a.legend()
plt.tight_layout(); plt.savefig("deletion_insertion_curves.png", dpi=200); plt.show()


In [ ]:
fig, axs = plt.subplots(len(CLASSES), 4, figsize=(13, 3.2 * len(CLASSES)))
for row, c in enumerate(CLASSES):
    r = eval_df[eval_df["label"] == c].iloc[0]
    raw = load_img(r["path"]).numpy().astype("uint8")
    x = preprocess_input(load_img(r["path"]).numpy())
    cls = int(np.argmax(model(x[None], training=False)[0]))
    gc, ig = gradcam(x[None], cls), integrated_gradients(x[None], cls)
    def overlay(m):
        heat = cv2.applyColorMap((m * 255).astype("uint8"), cv2.COLORMAP_JET)[..., ::-1]
        return (0.55 * raw + 0.45 * heat).astype("uint8")
    for a, im, t in zip(axs[row], [raw, overlay(gc), overlay(ig), lesion_mask(r["mask_path"])],
                        [f"True: {c} | Pred: {CLASSES[cls]}", "Grad-CAM", "Integrated Gradients", "Lesion mask (GT)"]):
        a.imshow(im, cmap="gray"); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.savefig("xai_qualitative.png", dpi=200); plt.show()